# FinChart-R2 - Phase 2C: Train Preference Mining for DPO (Kaggle)

This Kaggle notebook runs the Hugging Face adapter Kxck/Finance_500_v1 on unseen ChartQA train examples, writes JSONL prediction queues to Kaggle output, and prepares candidates for teacher review and later multimodal DPO.

It has two inference backends:

- TRANSFORMERS: default, robust Qwen3-VL + Unsloth path.
- VLLM: optional batched path. It is faster only when the Kaggle GPU and installed vLLM version can load the Qwen3-VL LoRA adapter successfully.

Research boundary: this notebook reads ChartQA train only. It never evaluates ChartQA val[0:500].


## 1. Kaggle settings

In the right-side Notebook Settings panel, enable a GPU and Internet. Internet is required to download ChartQA and the public Hugging Face adapter. Output files are written under /kaggle/working and persist after Save Version.


In [ ]:
import os
import sys
import subprocess

# Change to VLLM only after the Transformers smoke test has worked at least once.
ENGINE_REQUESTED = 'TRANSFORMERS'  # 'TRANSFORMERS' or 'VLLM'
assert ENGINE_REQUESTED in {'TRANSFORMERS', 'VLLM'}

if not os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    print('Warning: this notebook is intended for Kaggle.')
print('Python:', sys.version)
print('Requested backend:', ENGINE_REQUESTED)


In [ ]:
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages])

pip_install(
    '-U',
    'unsloth',
    'unsloth_zoo',
    'transformers',
    'accelerate',
    'bitsandbytes',
    'peft',
    'datasets',
    'tqdm',
    'huggingface_hub',
)

if ENGINE_REQUESTED == 'VLLM':
    pip_install('-U', 'vllm>=0.11.0', 'qwen-vl-utils==0.0.14')

print('Dependencies installed. If Kaggle asks for a session restart, restart once before continuing.')


## 2. Paths, Hugging Face access, and reproducible run configuration


In [ ]:
from pathlib import Path

WORK_DIR = Path('/kaggle/working/finchart_r2_phase2c_train_mining')
WORK_DIR.mkdir(parents=True, exist_ok=True)

SFT_ADAPTER_ID = 'Kxck/Finance_500_v1'
TRANSFORMERS_BASE_MODEL = 'unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit'
VLLM_BASE_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
DATASET_NAME = 'HuggingFaceM4/ChartQA'

# Keep the mine outside the original Phase 2A pilot pool.
TRAIN_START = 500
MINE_N = 2000
MAX_NEW_TOKENS = 64
VLLM_BATCH_SIZE = 8
SAVE_EVERY = 25

RUN_TAG = f'train_{TRAIN_START}_{TRAIN_START + MINE_N}'
ALL_PATH = WORK_DIR / f'phase2c_{RUN_TAG}_sft_predictions.jsonl'
ERROR_PATH = WORK_DIR / f'phase2c_{RUN_TAG}_sft_errors.jsonl'
CORRECT_PATH = WORK_DIR / f'phase2c_{RUN_TAG}_sft_correct.jsonl'
MANIFEST_PATH = WORK_DIR / f'phase2c_{RUN_TAG}_manifest.json'

# Optional: add a Kaggle Secret called HF_TOKEN for higher Hugging Face limits.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

print('Output directory:', WORK_DIR)
print('Using HF token:', bool(HF_TOKEN))


## 3. Load ChartQA train only and resume from JSONL


In [ ]:
import json
from datasets import load_dataset

chartqa_train = load_dataset(DATASET_NAME, split='train', token=HF_TOKEN)
assert TRAIN_START >= 500, 'Keep mining outside the original Phase 2A pilot pool.'
assert TRAIN_START + MINE_N <= len(chartqa_train), 'Requested train range is out of bounds.'

def read_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

existing_rows = read_jsonl(ALL_PATH)
completed_indices = {int(row['dataset_index']) for row in existing_rows}
expected_indices = set(range(TRAIN_START, TRAIN_START + MINE_N))
assert completed_indices <= expected_indices, 'Existing output belongs to a different run range.'

print('ChartQA train size:', len(chartqa_train))
print('Mining range:', TRAIN_START, 'to', TRAIN_START + MINE_N - 1)
print('Completed:', len(completed_indices), 'Remaining:', MINE_N - len(completed_indices))


## 4. Common deterministic matcher and final-answer prompt


In [ ]:
import re
import string

def unwrap_answer(value):
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return value[0]
    return value

def normalize_answer(text):
    text = '' if text is None else str(text)
    text = text.lower().strip()
    text = re.sub(r'^\s*final\s+answer\s*:\s*', '', text)
    text = re.sub(r'^\s*answer\s*:\s*', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*/\s*', '/', text)
    return text.strip(string.whitespace + '.,;:!?')

def try_parse_number(text):
    try:
        return float(normalize_answer(text).replace(',', '').replace('%', '').strip())
    except (ValueError, TypeError):
        return None

def deterministic_match(prediction, ground_truth, tolerance=1e-6):
    pred_norm = normalize_answer(prediction)
    gt_norm = normalize_answer(ground_truth)
    if pred_norm == gt_norm:
        return True
    pred_num = try_parse_number(pred_norm)
    gt_num = try_parse_number(gt_norm)
    return pred_num is not None and gt_num is not None and abs(pred_num - gt_num) <= tolerance

def question_text(question):
    return (
        'Look carefully at the chart and answer the question.\n\n'
        f'Question: {question}\n\n'
        'Return only the final answer.'
    )


## 5. Load adapter from Hugging Face and test one chart

The vLLM path is optional. If it cannot load the multimodal LoRA adapter on the assigned Kaggle GPU, the notebook automatically falls back to Transformers/Unsloth and records that choice in the manifest.


In [ ]:
ENGINE_USED = ENGINE_REQUESTED
ENGINE_FALLBACK_REASON = None

def load_transformers_backend():
    import unsloth
    import torch
    from peft import PeftModel
    from unsloth import FastVisionModel

    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
    model, processor = FastVisionModel.from_pretrained(
        model_name=TRANSFORMERS_BASE_MODEL,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(model, SFT_ADAPTER_ID, token=HF_TOKEN)
    FastVisionModel.for_inference(model)
    return model, processor

if ENGINE_REQUESTED == 'VLLM':
    try:
        from huggingface_hub import snapshot_download
        from transformers import AutoProcessor
        from vllm import LLM, SamplingParams
        from vllm.lora.request import LoRARequest

        adapter_local = snapshot_download(repo_id=SFT_ADAPTER_ID, token=HF_TOKEN)
        vllm_processor = AutoProcessor.from_pretrained(VLLM_BASE_MODEL, token=HF_TOKEN)
        llm = LLM(
            model=VLLM_BASE_MODEL,
            dtype='float16',
            max_model_len=2048,
            gpu_memory_utilization=0.85,
            enable_lora=True,
            max_lora_rank=64,
        )
        lora_request = LoRARequest('finance_500_v1', 1, adapter_local)
        sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)
        model = None
        processor = None
    except Exception as exc:
        ENGINE_USED = 'TRANSFORMERS'
        ENGINE_FALLBACK_REASON = f'vLLM initialization failed: {type(exc).__name__}: {exc}'
        print(ENGINE_FALLBACK_REASON)
        model, processor = load_transformers_backend()
else:
    model, processor = load_transformers_backend()

print('Inference backend:', ENGINE_USED)


In [ ]:
import torch

def predict_transformers(image, question):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': question_text(question)},
        ],
    }]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=image, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    return processor.batch_decode(
        output_ids[:, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )[0].strip()

def predict_vllm_batch(items):
    requests = []
    for image, question in items:
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': question_text(question)},
            ],
        }]
        prompt = vllm_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        requests.append({
            'prompt': prompt,
            'multi_modal_data': {'image': image},
        })
    outputs = llm.generate(requests, sampling_params, lora_request=lora_request)
    return [output.outputs[0].text.strip() for output in outputs]

smoke = chartqa_train[TRAIN_START]
smoke_question = str(smoke.get('query', smoke.get('question', ''))).strip()
if ENGINE_USED == 'VLLM':
    smoke_prediction = predict_vllm_batch([(smoke['image'], smoke_question)])[0]
else:
    smoke_prediction = predict_transformers(smoke['image'], smoke_question)

print('Adapter smoke-test question:', smoke_question)
print('Adapter prediction:', smoke_prediction)


## 6. Mine train-only predictions with JSONL checkpoints


In [ ]:
from tqdm.auto import tqdm

def append_jsonl(path, row):
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

pending_indices = [
    index for index in range(TRAIN_START, TRAIN_START + MINE_N)
    if index not in completed_indices
]

def save_row(dataset_index, example, prediction):
    question = str(example.get('query', example.get('question', ''))).strip()
    ground_truth = str(unwrap_answer(example.get('label', example.get('answer', '')))).strip()
    is_correct = deterministic_match(prediction, ground_truth)
    append_jsonl(ALL_PATH, {
        'dataset_index': dataset_index,
        'image_dataset': DATASET_NAME,
        'image_split': 'train',
        'image_index': dataset_index,
        'question': question,
        'ground_truth': ground_truth,
        'sft_prediction': prediction,
        'ground_truth_normalized': normalize_answer(ground_truth),
        'prediction_normalized': normalize_answer(prediction),
        'deterministic_correct': is_correct,
        'candidate_status': 'MODEL_CORRECT' if is_correct else 'CANDIDATE_INCORRECT',
        'source_adapter': SFT_ADAPTER_ID,
        'inference_backend': ENGINE_USED,
        'prompt_protocol': 'phase1_final_answer_v1',
    })

if ENGINE_USED == 'VLLM':
    iterator = range(0, len(pending_indices), VLLM_BATCH_SIZE)
    for offset in tqdm(iterator, desc='vLLM train mining'):
        batch_indices = pending_indices[offset:offset + VLLM_BATCH_SIZE]
        batch_examples = [chartqa_train[index] for index in batch_indices]
        items = [
            (example['image'], str(example.get('query', example.get('question', ''))).strip())
            for example in batch_examples
        ]
        predictions = predict_vllm_batch(items)
        for dataset_index, example, prediction in zip(batch_indices, batch_examples, predictions):
            save_row(dataset_index, example, prediction)
else:
    for position, dataset_index in enumerate(tqdm(pending_indices, desc='Transformers train mining'), start=1):
        example = chartqa_train[dataset_index]
        question = str(example.get('query', example.get('question', ''))).strip()
        prediction = predict_transformers(example['image'], question)
        save_row(dataset_index, example, prediction)
        if position % SAVE_EVERY == 0:
            print(f'Checkpoint: {position}/{len(pending_indices)} new samples')

print('Mining complete:', ALL_PATH)


## 7. Export queues and manifest for the teacher/DPO stage


In [ ]:
all_rows = read_jsonl(ALL_PATH)
assert len({row['dataset_index'] for row in all_rows}) == MINE_N, 'Run is incomplete; rerun Cell 6 to resume.'

errors = [row for row in all_rows if not row['deterministic_correct']]
correct = [row for row in all_rows if row['deterministic_correct']]

def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')

write_jsonl(ERROR_PATH, errors)
write_jsonl(CORRECT_PATH, correct)
manifest = {
    'dataset': DATASET_NAME,
    'source_split': 'train',
    'train_start': TRAIN_START,
    'mine_n': MINE_N,
    'source_adapter': SFT_ADAPTER_ID,
    'engine_requested': ENGINE_REQUESTED,
    'engine_used': ENGINE_USED,
    'engine_fallback_reason': ENGINE_FALLBACK_REASON,
    'total_predictions': len(all_rows),
    'deterministic_correct': len(correct),
    'candidate_incorrect': len(errors),
    'outputs': {
        'all': str(ALL_PATH),
        'errors': str(ERROR_PATH),
        'correct': str(CORRECT_PATH),
    },
    'next_step': 'Teacher audit of errors only, then schema-matched train-only DPO pairs.',
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Save Version in Kaggle to persist /kaggle/working outputs.')


## Handoff

Use the saved error JSONL as a teacher-review queue only. Do not train raw predictions directly. Validated DPO pairs must use identical response schemas in chosen and rejected and remain sourced from ChartQA train.
